In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

# Setup

In [15]:
TRAIN_MODE   = 1
INF_MODE     = 1 # Inference / Predication Mode 
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
UPLOAD_MODEL = {'upload': TRAIN_MODE, 'file_name': 'model_t1', 'url': "jashmevada/dlgenai-nppe2/pytorch/cnn_lstm"}
LOAD_MODEL   = 0
EPOCHS       = 100
TRACKIO_NAME = "cnn-lstm_8"
MODEL_TYPE   = 2 # 0 -> RNN ; 1 -> bi-LSTM; 2 -> CNN-LSTM

In [3]:
!pip install trackio -qq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 988.0/988.0 kB 22.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 102.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.4/55.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 120.0 MB/s eta 0:00:0000:0100:01


In [4]:
from kaggle_secrets import UserSecretsClient
import os
user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")

In [5]:
import trackio
if TRAIN_MODE: trackio.init(project="25-t3-nppe2", space_id="JashMevada/dlgenai-nppe", name=TRACKIO_NAME, group="cnn_lstm_model") # 25-t3-nppe2

* Trackio project initialized: test2
* Trackio metrics will be synced to Hugging Face Dataset: JashMevada/dlgenai-nppe-dataset
* Found existing space: https://huggingface.co/spaces/JashMevada/dlgenai-nppe
* View dashboard by going to: https://JashMevada-dlgenai-nppe.hf.space/


* Created new run: cnn-lstm_7


In [6]:
df = pd.read_csv("/kaggle/input/sep-25-dl-gen-ai-nppe-2/train.csv")
test_df = pd.read_csv("/kaggle/input/sep-25-dl-gen-ai-nppe-2/test.csv")

# Models

In [7]:
import torch
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from sklearn.model_selection import train_test_split

AA_MAP = {aa: i+1 for i, aa in enumerate("ACDEFGHIKLMNPQRSTVWY")} # Standard 20
AA_MAP['*'] = 21 # Masked
PAD_TOKEN = 0
AA_MAP['<PAD>'] = PAD_TOKEN

Q3_MAP = {l: i+1 for i, l in enumerate("HEC")}
Q3_MAP['<PAD>'] = 0

# Q8 Map
Q8_MAP = {l: i+1 for i, l in enumerate("HGIEBTSC")}
Q8_MAP['<PAD>'] = 0

def encode_seq(seq, mapping):
    return [mapping.get(c, mapping.get('*', 0)) for c in seq]

class ProteinDataset(Dataset):
    def __init__(self, df, max_len=512, mode='train'):
        self.df = df
        self.max_len = max_len
        self.mode = mode
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        seq = row['seq']
        encoded_seq = encode_seq(seq, AA_MAP)
        
        # Padding/Truncating
        if len(encoded_seq) > self.max_len:
            encoded_seq = encoded_seq[:self.max_len]
        else:
            encoded_seq = encoded_seq + [PAD_TOKEN] * (self.max_len - len(encoded_seq))
            
        item = {
            'seq': torch.tensor(encoded_seq, dtype=torch.long),
            'mask': torch.tensor([1 if t != PAD_TOKEN else 0 for t in encoded_seq], dtype=torch.long) # Attention mask
        }
        
        if self.mode == 'train':
            q3 = row['sst3']
            q8 = row['sst8']
            
            enc_q3 = encode_seq(q3, Q3_MAP)
            enc_q8 = encode_seq(q8, Q8_MAP)
            
            # Pad Labels
            if len(enc_q3) > self.max_len:
                enc_q3 = enc_q3[:self.max_len]
                enc_q8 = enc_q8[:self.max_len]
            else:
                enc_q3 = enc_q3 + [0] * (self.max_len - len(enc_q3))
                enc_q8 = enc_q8 + [0] * (self.max_len - len(enc_q8))
                
            item['q3'] = torch.tensor(enc_q3, dtype=torch.long)
            item['q8'] = torch.tensor(enc_q8, dtype=torch.long)
            
        return item

class ProteinDataModule(pl.LightningDataModule):
    def __init__(self, train_df, test_df, batch_size=32, max_len=512):
        super().__init__()
        self.train_df = train_df
        self.test_df = test_df
        self.batch_size = batch_size
        self.max_len = max_len
        
    def setup(self, stage=None): 
        if 'seq_len' not in self.train_df.columns:
            self.train_df['seq_len'] = self.train_df['seq'].apply(len)
            
        try:
            self.train_df['len_bin'] = pd.qcut(self.train_df['seq_len'], q=10, labels=False, duplicates='drop')
        except ValueError:
             # Fallback if qcut fails
             self.train_df['len_bin'] = pd.cut(self.train_df['seq_len'], bins=5, labels=False)

        self.train_data, self.val_data = train_test_split(
            self.train_df, 
            test_size=0.15, 
            random_state=42, 
            stratify=self.train_df['len_bin']
        )
        
        print(f"Training Set: {len(self.train_data)} samples")
        print(f"Validation Set: {len(self.val_data)} samples")
        
        self.train_ds = ProteinDataset(self.train_data, self.max_len, mode='train')
        self.val_ds = ProteinDataset(self.val_data, self.max_len, mode='train')
        self.test_ds = ProteinDataset(self.test_df, self.max_len, mode='test')
        
    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=self.batch_size, shuffle=True, num_workers=3)
        
    def val_dataloader(self):
        return DataLoader(self.val_ds, batch_size=self.batch_size, num_workers=3)

    def test_dataloader(self):
        return DataLoader(self.test_ds, batch_size=self.batch_size, num_workers=3)

max_train_len = df['seq'].str.len().max()
max_test_len = test_df['seq'].str.len().max()
MAX_LEN = max(max_train_len, max_test_len) + 10 # Add buffer
print(f"Max sequence length found: {MAX_LEN}")

data_module = ProteinDataModule(df, test_df, batch_size=32, max_len=MAX_LEN)

Max sequence length found: 1642


In [10]:
import torch.nn as nn
import torch.nn.functional as F
from torchmetrics.classification import MulticlassF1Score
from torch.optim.lr_scheduler import ReduceLROnPlateau

class BaseProteinModel(pl.LightningModule):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim_q3, output_dim_q8, lr=2e-3, label_smoothing=0.0):
        super().__init__()
        self.save_hyperparameters()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.label_smoothing = label_smoothing
        
        # Metrics
        self.f1_q3 = MulticlassF1Score(num_classes=output_dim_q3, ignore_index=0, average='macro')
        self.f1_q8 = MulticlassF1Score(num_classes=output_dim_q8, ignore_index=0, average='macro')

    def forward(self, x, mask):
        raise NotImplementedError

    def training_step(self, batch, batch_idx):
        seq = batch['seq']
        mask = batch['mask'].float()
        q3_target = batch['q3']
        q8_target = batch['q8']

        q3_logits, q8_logits = self(seq, batch['mask'])

        # Reshape for loss: (N, C, L) vs (N, L)
        loss_q3_tokens = F.cross_entropy(q3_logits.transpose(1, 2), q3_target, reduction="none", ignore_index=0,label_smoothing=self.label_smoothing)
        loss_q8_tokens = F.cross_entropy(q8_logits.transpose(1, 2), q8_target, reduction="none", ignore_index=0,label_smoothing=self.label_smoothing)

        denom = mask.sum() + 1e-8
        loss_q3 = (loss_q3_tokens * mask).sum() / denom
        loss_q8 = (loss_q8_tokens * mask).sum() / denom
        
        loss = loss_q3 + loss_q8
        self.log('train_loss', loss, prog_bar=True)
        trackio.log({'train_loss': loss.item()})

        return loss

    def validation_step(self, batch, batch_idx):
        seq = batch['seq']
        mask = batch['mask'].float()
        q3_target = batch['q3']
        q8_target = batch['q8']

        q3_logits, q8_logits = self(seq, batch['mask'])

        loss_q3_tokens = F.cross_entropy(q3_logits.transpose(1, 2), q3_target, reduction="none", ignore_index=0,label_smoothing=self.label_smoothing)
        loss_q8_tokens = F.cross_entropy(q8_logits.transpose(1, 2), q8_target, reduction="none", ignore_index=0,label_smoothing=self.label_smoothing)

        denom = mask.sum() + 1e-8
        loss_q3 = (loss_q3_tokens * mask).sum() / denom
        loss_q8 = (loss_q8_tokens * mask).sum() / denom
        
        # Calculate F1
        self.f1_q3(q3_logits.transpose(1, 2), q3_target)
        self.f1_q8(q8_logits.transpose(1, 2), q8_target)

        self.log('val_loss', loss_q3 + loss_q8, prog_bar=True)
        self.log('val_f1_q3', self.f1_q3, on_step=False, on_epoch=True, prog_bar=True)
        self.log('val_f1_q8', self.f1_q8, on_step=False, on_epoch=True, prog_bar=True)

        trackio.log({
            'val_loss': loss_q3.item() + loss_q8.item(),
            'val_f1_q3': self.f1_q3.compute().item(),
            'val_f1_q8': self.f1_q8.compute().item(),
        })

    def on_validation_epoch_end(self):
        # Compute harmonic mean of F1 scores
        f1_q3 = self.f1_q3.compute()
        f1_q8 = self.f1_q8.compute()
        score = 2 * f1_q3 * f1_q8 / (f1_q3 + f1_q8 + 1e-8)
        
        self.log('val_score', score, prog_bar=True)
        trackio.log({'val_score': score.item()})

    def predict_step(self, batch, batch_idx):
        seq = batch['seq']
        mask = batch['mask']
        q3_logits, q8_logits = self(seq, mask)
        
        return {
            'q3_preds': torch.argmax(q3_logits, dim=-1),
            'q8_preds': torch.argmax(q8_logits, dim=-1)
        }

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.lr)
        scheduler = ReduceLROnPlateau(optimizer, factor=0.5, patience=3, mode='max', min_lr = 1e-5)
        
        return {"optimizer": optimizer, "lr_scheduler": {"scheduler": scheduler, "monitor": "val_score"}}

class BiRNNModel(BaseProteinModel):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim_q3, output_dim_q8, num_layers=2):
        super().__init__(vocab_size, embedding_dim, hidden_dim, output_dim_q3, output_dim_q8)
        
        self.rnn = nn.RNN(embedding_dim, hidden_dim, num_layers=num_layers, 
                          batch_first=True, bidirectional=True, dropout=0.1)
        
        # Output heads
        self.fc_q3 = nn.Linear(hidden_dim * 2, output_dim_q3)
        self.fc_q8 = nn.Linear(hidden_dim * 2, output_dim_q8)
        
    def forward(self, x, mask):
        embedded = self.embedding(x)
        out, _ = self.rnn(embedded)
        
        q3_out = self.fc_q3(out)
        q8_out = self.fc_q8(out)
        return q3_out, q8_out

class BiLSTMModel(BaseProteinModel):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim_q3, output_dim_q8, num_layers=2):
        super().__init__(vocab_size, embedding_dim, hidden_dim, output_dim_q3, output_dim_q8)
        
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers, 
                          batch_first=True, bidirectional=True, dropout=0.1)
        
        self.fc_q3 = nn.Linear(hidden_dim * 2, output_dim_q3)
        self.fc_q8 = nn.Linear(hidden_dim * 2, output_dim_q8)
        
    def forward(self, x, mask):
        embedded = self.embedding(x)
        out, _ = self.lstm(embedded)
        
        q3_out = self.fc_q3(out)
        q8_out = self.fc_q8(out)
        return q3_out, q8_out

class CNNBiLSTMModel(BaseProteinModel):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim_q3, output_dim_q8, kernal_sizes=[1,3,5,7],num_layers=2, label_smoothing=0.0, dropout=0.3):
        super().__init__(vocab_size, embedding_dim, hidden_dim, output_dim_q3, output_dim_q8, label_smoothing=label_smoothing)

        self.conv1 = nn.Conv1d(in_channels=embedding_dim, out_channels=hidden_dim, kernel_size=1, padding=0)
        self.conv3 = nn.Conv1d(in_channels=embedding_dim, out_channels=hidden_dim, kernel_size=3, padding=1)
        self.conv5 = nn.Conv1d(in_channels=embedding_dim, out_channels=hidden_dim, kernel_size=5, padding=2)
        self.conv7 = nn.Conv1d(in_channels=embedding_dim, out_channels=hidden_dim, kernel_size=7, padding=3)

        self.dropout_conv = nn.Dropout(dropout)
        self.relu = nn.ReLU()

        self.lstm = nn.LSTM(input_size=hidden_dim * 4, hidden_size=hidden_dim,
                            num_layers=num_layers, batch_first=True,
                            bidirectional=True, dropout=0.1)
        self.dropout_lstm = nn.Dropout(dropout)

        self.q3_proj = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim * 2),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.fc_q3 = nn.Linear(hidden_dim * 2, output_dim_q3)

        self.q8_proj = nn.Sequential(
            nn.Linear(hidden_dim * 2 + output_dim_q3, hidden_dim * 2),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        
        self.fc_q8 = nn.Linear(hidden_dim * 2, output_dim_q8)

    def forward(self, x, mask):
        embedded = self.embedding(x) # (Batch, Seq_Len, Emb_Dim)

        embedded = embedded.permute(0, 2, 1)

        c1 = self.relu(self.conv1(embedded))
        c3 = self.relu(self.conv3(embedded))
        c5 = self.relu(self.conv5(embedded))
        c7 = self.relu(self.conv7(embedded))
        
        lstm_in = torch.cat([c1,c3,c5,c7], dim=1)
        lstm_in = self.dropout_conv(lstm_in)
        lstm_in = lstm_in.permute(0, 2, 1)
        
        lstm_out, _ = self.lstm(lstm_in)
        lstm_out = self.dropout_lstm(lstm_out)

        q3_hidden = self.q3_proj(lstm_out)
        q3_logits = self.fc_q3(q3_hidden)

        with torch.no_grad():
            q3_probs = F.softmax(q3_logits, dim=-1)    # (B, L, C3), used as feature only
        
        q8_input = torch.cat([lstm_out, q3_probs], dim=-1)
        q8_hidden = self.q8_proj(q8_input)
        q8_logits = self.fc_q8(q8_hidden) # lstm_out 
        
        return q3_logits, q8_logits

In [16]:
VOCAB_SIZE = len(AA_MAP) + 1 
EMBEDDING_DIM = 64
HIDDEN_DIM = 256
OUTPUT_DIM_Q3 = len(Q3_MAP) + 1
OUTPUT_DIM_Q8 = len(Q8_MAP) + 1

model_rnn = BiRNNModel(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, OUTPUT_DIM_Q3, OUTPUT_DIM_Q8)
model_lstm = BiLSTMModel(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, OUTPUT_DIM_Q3, OUTPUT_DIM_Q8)
model_cnn_lstm = CNNBiLSTMModel(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, OUTPUT_DIM_Q3, OUTPUT_DIM_Q8, label_smoothing=0.1)

early_stopping = pl.callbacks.EarlyStopping(monitor="val_score", mode="max", patience=3,verbose=True)

trainer = pl.Trainer(max_epochs=EPOCHS, accelerator="auto", devices=1, log_every_n_steps=50,callbacks=[early_stopping])

if TRAIN_MODE == 1  and MODEL_TYPE == 0: 
    trainer.fit(model_rnn, data_module)
    trackio.finish()

if TRAIN_MODE == 1  and MODEL_TYPE == 1: 
    trainer.fit(model_lstm, data_module)
    trackio.finish()
    
if TRAIN_MODE == 1  and MODEL_TYPE == 2: 
    trainer.fit(model_cnn_lstm, data_module)
    trackio.finish()

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

   | Name         | Type              | Params | Mode 
------------------------------------------------------------
0  | embedding    | Embedding         | 1.5 K  | train
1  | f1_q3        | MulticlassF1Score | 0      | train
2  | f1_q8        | MulticlassF1Score | 0      | train
3  | conv1        | Conv1d            | 16.6 K | train
4  | conv3        | Conv1d            | 49.4 K | train
5  | conv5        | Conv1d            | 82.2 K | train
6  | conv7        | Conv1d            | 114 K  | train
7  | dropout_conv | Dropout           | 0      | train
8  | relu         | ReLU              | 0      | train
9  | lstm         | LSTM 

Training Set: 6172 samples
Validation Set: 1090 samples


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Metric val_score improved. New best score: 0.387


Validation: |          | 0/? [00:00<?, ?it/s]

Metric val_score improved by 0.020 >= min_delta = 0.0. New best score: 0.407
`Trainer.fit` stopped: `max_epochs=2` reached.


* Run finished. Uploading logs to Trackio (please wait...)


# Submission

In [ ]:
import kagglehub
def save_model(model):
    torch.save(model.state_dict(), f"/kaggle/working/{UPLOAD_MODEL['file_name']}.pth")
    kagglehub.model_upload(UPLOAD_MODEL['url'], f"/kaggle/working/{UPLOAD_MODEL['file_name']}.pth") 

try:
    if UPLOAD_MODEL['upload']:
        if MODEL_TYPE == 0: save_model(model_rnn)
        if MODEL_TYPE == 1: save_model(model_lstm)
        if MODEL_TYPE == 2: save_model(model_cnn_lstm)
            
except Exception as e:
    print("An exception occurred at Save model")
    print("Error: ", e)

In [17]:
import kagglehub

def create_submission(model, data_module: pl.LightningDataModule, test_df, output_path="submission.csv"):
    if LOAD_MODEL:
        try:
            path = kagglehub.model_download(UPLOAD_MODEL['url'])
            model_path = f"{path}/{UPLOAD_MODEL['file_name']}.pth"
            if not os.path.exists(model_path):
                 model_path = f"{path}/{UPLOAD_MODEL['file_name']}.pth"
            
            state_dict = torch.load(model_path, map_location=DEVICE)
            model.load_state_dict(state_dict)
            print(f"Model loaded from {model_path}")
        except Exception as e:
            print(f"Warning: Could not load model from KaggleHub: {e}")
            print("Proceeding with current model weights...")

    model.eval()
    model.to(DEVICE)
    
    data_module.setup()
    
    trainer = pl.Trainer(accelerator="auto", devices=1, logger=False)
    predictions = trainer.predict(model, dataloaders=data_module.test_dataloader())
    
    q3_preds_all = []
    q8_preds_all = []
    
    for batch in predictions:
        q3_preds_all.extend(batch['q3_preds'].cpu().numpy())
        q8_preds_all.extend(batch['q8_preds'].cpu().numpy())
        
    # Inverse Maps
    INV_Q3_MAP = {v: k for k, v in Q3_MAP.items()}
    INV_Q8_MAP = {v: k for k, v in Q8_MAP.items()}
    
    final_q3 = []
    final_q8 = []
    
    print("Decoding predictions...")
    for i, row in test_df.iterrows():
        seq_len = len(row['seq'])
        
        # Get preds and truncate to seq_len
        # Note: predictions are padded to max_len of the batch or dataset
        p3 = q3_preds_all[i][:seq_len]
        p8 = q8_preds_all[i][:seq_len]
        
        # Decode
        s3 = "".join([INV_Q3_MAP.get(p, 'C') for p in p3]) 
        s8 = "".join([INV_Q8_MAP.get(p, 'C') for p in p8]) 

        final_q3.append(s3)
        final_q8.append(s8)
        
    submission_df = pd.DataFrame({
        'id': test_df['id'],
        'sst8': final_q8,
        'sst3': final_q3
    })
    
    submission_df.to_csv(output_path, index=False)
    print(f"Submission file saved to {output_path}")
    print(submission_df.head())

if INF_MODE:
    if MODEL_TYPE == 0:
        create_submission(model_rnn, data_module, test_df)
    elif MODEL_TYPE == 1:
        create_submission(model_lstm, data_module, test_df)
    elif MODEL_TYPE == 2:
        create_submission(model_cnn_lstm, data_module, test_df)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Training Set: 6172 samples
Validation Set: 1090 samples


Predicting: |          | 0/? [00:00<?, ?it/s]

Decoding predictions...
Submission file saved to submission.csv
   id                                               sst8  \
0   0  CCCCCHHHHHHHHHHHHHHHHHHCCEEEEEECCCTTCCCEEEEECT...   
1   1  CCCCCCCCCCCEEEEEECTTTEEEEEEECTTHHHHHHHHCCCTTHH...   
2   2  CCHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHCCHHHHHHHHH...   
3   3  CCCCCCHHHCHHHHHHHHHHHHHHHHHHHCCHHHHHHHHHHHHHHH...   
4   4  CEEEEEEECTCTHHHHHHHHHHHHHHHCEEEECCTTTEEEEECSEE...   

                                                sst3  
0  CCCCCCHHHHHHHHHHHHHHHHHCCEEEEEECCCCCCCCEEEEECC...  
1  CCCCCCCCCCCEEEEEECCCCEEEEEEECCCHHHHHHHHCCCCCHH...  
2  CCHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHCCCHHHHHHHH...  
3  CCCCCCHHHCCCHHHHHHHHHHHHHHHHHCCCHHHHHHHHHHHHHH...  
4  CEEEEEEECCCCHHHHHHHHHHHHHHHCCCCCCCCCCEEEEECCCE...  
